# Week 54, Day 4: Bias and variance, measured

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w054/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Three terms, measured

For each degree, stack the predictions of the 300 fits on the grid into an array of shape (300, len(grid)). Compute bias squared (the squared gap between the average prediction and the truth, averaged over the grid) and variance (the variance across fits, averaged over the grid). Compare their sum plus the noise with the test error measured directly against fresh noisy targets.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(544)
f = lambda x: 0.8 * np.sin(np.pi * x)
sigma, n, reps = 0.3, 30, 300
grid = np.linspace(-0.95, 0.95, 200)
sets = [(xt := r.uniform(-1, 1, n), f(xt) + sigma * r.normal(size=n)) for _ in range(reps)]
bias2, var, test_mse = np.zeros(8), np.zeros(8), np.zeros(8)
for deg in range(8):
    preds = np.array([np.polyval(np.polyfit(xt, yt, deg), grid) for xt, yt in sets])
    mean_pred = preds.mean(0)
    bias2[deg] = ...
    var[deg] = ...
    y_new = f(grid) + sigma * r.normal(size=(reps, len(grid)))
    test_mse[deg] = np.mean((y_new - preds) ** 2)
for d in range(8):
    print(f"degree {d}: bias² {bias2[d]:.4f}, variance {var[d]:.4f}, sum with noise {sigma ** 2 + bias2[d] + var[d]:.4f}, measured {test_mse[d]:.4f}")
plt.plot(bias2, "o-", label="bias²"); plt.plot(var, "o-", label="variance"); plt.plot(sigma ** 2 + bias2 + var, "k--", label="total")
plt.xlabel("degree"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_terms():
    "bias2 and var have one value per degree, all non-negative"
    assert bias2.shape == (8,) and var.shape == (8,) and np.all(bias2 >= 0) and np.all(var >= 0)
def test_trends():
    "bias falls and variance rises from degree 1 to degree 7"
    assert bias2[1] > bias2[7] and var[7] > var[1]
def test_sum():
    "noise plus bias squared plus variance matches the directly simulated test error within 5%"
    assert np.all(np.abs(sigma ** 2 + bias2 + var - test_mse) / test_mse < 0.05)

test_terms()
test_trends()
test_sum()
print("All checks passed")

### Exercise: Trading variance for bias

Write design(x), the matrix of powers 0 to 9. For each lambda, fit ridge (penalising every coefficient except the constant) to each of the 300 training sets, and record the bias squared and variance on the grid.

In [ ]:
def design(x):
    return ...

lams = np.logspace(-6, 1, 15)
D = np.eye(10); D[0, 0] = 0
G = design(grid)
bias_r, var_r = np.zeros(len(lams)), np.zeros(len(lams))
for k, lam in enumerate(lams):
    preds = np.array([G @ np.linalg.solve(design(xt).T @ design(xt) + lam * D, design(xt).T @ yt) for xt, yt in sets])
    bias_r[k] = np.mean((preds.mean(0) - f(grid)) ** 2)
    var_r[k] = np.mean(preds.var(axis=0))
k = np.argmin(bias_r + var_r)
print(f"lambda {lams[0]:.0e}: bias² {bias_r[0]:.4f}, variance {var_r[0]:.4f}")
print(f"lambda {lams[k]:.0e} (best): bias² {bias_r[k]:.4f}, variance {var_r[k]:.4f}")
print(f"lambda {lams[-1]:.0e}: bias² {bias_r[-1]:.4f}, variance {var_r[-1]:.4f}")
plt.loglog(lams, bias_r, "o-", label="bias²"); plt.loglog(lams, var_r, "o-", label="variance"); plt.xlabel("lambda"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_design():
    "design(x) returns the 10 polynomial features 1, x, ..., x^9"
    assert np.allclose(design(np.array([2.0]))[0], 2.0 ** np.arange(10))
def test_dial():
    "as lambda grows, variance falls and bias rises"
    assert var_r[0] > var_r[-1] and bias_r[-1] > bias_r[0]
def test_best():
    "the best lambda is in the interior and beats both ends"
    tot = bias_r + var_r; k = np.argmin(tot)
    assert 0 < k < len(lams) - 1

test_design()
test_dial()
test_best()
print("All checks passed")

### Exercise: Bagging a high-variance model

Nearest-neighbour regression, which predicts each input by the target of the closest training point, has low bias and high variance. Write nn_predict(xt, yt) on the grid, and bagged_fit(xt, yt, B, g), which averages nn_predict over B bootstrap resamples (indices drawn with replacement). Compare the variance and bias of single and bagged fits across the first 100 training sets.

In [ ]:
def nn_predict(xt, yt):
    return ...

def bagged_fit(xt, yt, B, g):
    preds = []
    for b in range(B):
        idx = g.integers(0, len(xt), len(xt))
        preds.append(...)
    return np.mean(preds, axis=0)

g = np.random.default_rng(58)
single = np.array([nn_predict(xt, yt) for xt, yt in sets[:100]])
bagged = np.array([bagged_fit(xt, yt, 30, g) for xt, yt in sets[:100]])
var_single, var_bag = np.mean(single.var(0)), np.mean(bagged.var(0))
b_single, b_bag = np.mean((single.mean(0) - f(grid)) ** 2), np.mean((bagged.mean(0) - f(grid)) ** 2)
print(f"single nearest-neighbour fit: variance {var_single:.4f}, bias² {b_single:.4f}")
print(f"bagged (30 resamples): variance {var_bag:.4f}, bias² {b_bag:.4f}")

In [ ]:
import numpy as np
def test_nn():
    "nn_predict predicts each grid point by its nearest training point's target"
    assert np.allclose(nn_predict(np.array([-0.5, 0.5]), np.array([1.0, 2.0]))[[0, -1]], [1.0, 2.0])
def test_bagged():
    "bagged_fit averages B nearest-neighbour fits to bootstrap resamples"
    xt, yt = sets[0]; p = bagged_fit(xt, yt, 5, np.random.default_rng(0))
    assert p.shape == grid.shape
def test_var():
    "bagging cuts the variance of nearest-neighbour regression by at least a third, with little change in bias"
    assert var_bag < 0.67 * var_single and b_bag < 0.01

test_nn()
test_bagged()
test_var()
print("All checks passed")